# Probe Cấu hình D — Qwen2.5-VL đọc chữ đậm

Chạy trên Colab (Runtime → Change runtime type → **T4 GPU**).

**Trước khi chạy:** sửa `DRIVE` ở Cell 1 cho khớp đường dẫn Drive thật.

Dữ liệu đọc thẳng từ Drive — không zip, không upload. Kết quả `preds.jsonl` ghi thẳng về Drive.

In [ ]:
# --- Cell 1: mount Drive + chốt GPU + cài đặt (~2 phút) ---
DRIVE = "/content/drive/MyDrive/Slot 1/Working Files"   # <-- sửa đúng 1 dòng này

from google.colab import drive
drive.mount("/content/drive")

# Chốt GPU TRƯỚC khi pip. Runtime CPU vẫn cài transformers ngon lành rồi chết ở
# `torch.cuda.get_device_name(0)` với "Torch not compiled with CUDA enabled" —
# trông như lỗi code nhưng thật ra là Hardware accelerator chưa đặt T4 GPU.
# Bắt sớm để báo đúng việc phải làm, và khỏi tốn 2 phút pip vô ích.
# ponytail: 4 dòng, đổi lấy việc khỏi đoán lại mỗi lần Colab thu hồi runtime.
import torch
assert torch.cuda.is_available(), (
    f"Runtime này KHÔNG có GPU (torch {torch.__version__}). "
    "Runtime → Change runtime type → Hardware accelerator: T4 GPU → Save, "
    "rồi Runtime → Restart session (đổi runtime type phải restart mới ăn), "
    "rồi chạy lại Cell 1. Nếu ô đó ĐÃ là T4 GPU mà vẫn lỗi này thì đã hết quota "
    "GPU free — chờ vài tiếng hoặc đổi tài khoản.")

!pip -q install "transformers>=4.49" accelerate qwen-vl-utils bitsandbytes

print(torch.__version__, torch.cuda.get_device_name(0))
print(f"{torch.cuda.get_device_properties(0).total_memory/2**30:.0f} GB VRAM, bf16={torch.cuda.is_bf16_supported()}")

In [ ]:
# --- Cell 2: trỏ vào thư mục trên Drive ---
import json, pathlib, sys

ROOT = pathlib.Path(DRIVE)
P_OUT = ROOT / "probe_d"
sys.path.insert(0, str(ROOT / "code"))          # probe_vlm_bold.py nằm ở đây

CODE = ROOT / "code" / "probe_vlm_bold.py"
assert CODE.exists(), f"không thấy probe_vlm_bold.py trong {ROOT/'code'}"
assert (P_OUT / "items.jsonl").exists(), f"không thấy items.jsonl trong {P_OUT}"

# Drive giữ bản cũ rất dai, và có HAI thứ có thể cũ độc lập: file .py và file
# .ipynb này. Chốt cả hai.
#
# ponytail: KHÔNG chép số schema vào cell. Bản trước hardcode `== 2` nằm ngay
# cạnh `NEED_CODE = 3`, nên khi module lên schema 3 thì CHÍNH CELL CŨ bắn vào
# dữ liệu mới và báo nhầm "items.jsonl là bản cũ" — trong khi dòng print ngay
# trên đã nói `schema 3`. Hằng số chép tay không tự bắt được chính nó.
# Giờ cả hai chốt nằm trong module; cell chỉ gọi.
import importlib
import probe_vlm_bold as P

P = importlib.reload(P)
P.OUT = P_OUT

got = getattr(P, "CODE_VERSION", 0)
assert got >= 8, (
    f"probe_vlm_bold.py trên Drive là BẢN CŨ (CODE_VERSION {got} < 8). "
    f"Đẩy lại {CODE} từ máy, rồi Runtime → Restart and run all.")

items = P.load_items()          # module tự chặn nếu items.jsonl sai schema
print(len(items), "item · schema", items[0]["schema"],
      "· code v", got, "· cell cần schema", P.SCHEMA)
print(sorted(items[0].keys()))


In [ ]:
# --- Cell 2b: SANITY — chẩn đoán dụng cụ đo (chạy TRƯỚC Cell 3) ---
# 10 ảnh x 3 câu hỏi x 2 cấu hình (bf16 và 4-bit). ~2 phút.
#
# Lần chạy 120 câu trước cho ra 120/120 chọn "A" — hằng số. Một trong hai:
#   * đọc được chữ nhưng không SO SÁNH được độ đậm  -> lỗi tầng so sánh
#   * không đọc nổi chữ trong ảnh                   -> lỗi tầng nhìn
# Câu doc_A/doc_B tách hai khả năng đó (nhãn A/B nay vẽ thẳng lên ảnh). Chạy cả
# bf16 để biết 4-bit có bào mất nét chữ không.
import argparse, importlib
import probe_vlm_bold as P

P = importlib.reload(P)
P.OUT = P_OUT
P.cmd_sanity(argparse.Namespace(quant4=True, n=10, both=True))

In [12]:
# --- Cell 2c: SANITY 2 — lỗi ở tầng SO SÁNH hay ở CỘT NHÃN? (chạy SAU 2b) ---
# Cùng 10 ảnh, KHÔNG cắt lại. Chỉ chạy bf16 (2b đã đo 4-bit giống bf16 10/10). ~1 phút.
#
# 2b cho biết: model CHÉP được chữ cả hai dải (tầng nhìn ổn) nhưng trả 'A' cho cả
# 10 ảnh. Ba biến thể dưới đây tách hai nghi phạm còn lại:
#   V0 = mốc so sánh (prompt cũ)
#   V1 = buộc chép-rồi-so -> ĐỔI kết quả  => lỗi tầng SO SÁNH, sửa bằng prompt
#   V2 = bỏ cột tên đơn vị -> ĐỔI kết quả => lỗi CỘT NHÃN (dải còn lẫn hàng khác
#        đậm hơn nên model trả lời đúng theo thứ nó nhìn thấy), sửa bằng cách cắt
#        bỏ cột nhãn khỏi canvas
# Không đổi cả hai => lỗi sâu hơn ở tầng nhìn-cục-bộ, lúc đó mới kết luận về model.
import argparse, importlib
import probe_vlm_bold as P

P = importlib.reload(P)
P.OUT = P_OUT
P.cmd_sanity2(argparse.Namespace(n=10))

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

[bf16] transformers 5.18.0 | dtype thật=torch.bfloat16
sanity2 [bf16]: 10 ảnh x 3 biến thể

    B-train-00780-q04 V1 chép: 'ĐÁP ÁN: A'
[bf16] B-train-00780-q04 gold=1  V0_goc=0 X  V1_tung_buoc=0 X  V2_bo_cot_dau=0 X
    B-train-00212-q01 V1 chép: 'ĐÁP ÁN: A'
[bf16] B-train-00212-q01 gold=1  V0_goc=0 X  V1_tung_buoc=0 X  V2_bo_cot_dau=0 X
    B-train-00594-q02 V1 chép: 'ĐÁP ÁN: A'
[bf16] B-train-00594-q02 gold=1  V0_goc=0 X  V1_tung_buoc=0 X  V2_bo_cot_dau=0 X
    B-train-00148-q04 V1 chép: 'ĐÁP ÁN: A'
[bf16] B-train-00148-q04 gold=1  V0_goc=0 X  V1_tung_buoc=0 X  V2_bo_cot_dau=0 X
    B-train-00576-q01 V1 chép: 'ĐÁP ÁN: A'
[bf16] B-train-00576-q01 gold=1  V0_goc=0 X  V1_tung_buoc=0 X  V2_bo_cot_dau=0 X
    B-train-00238-q08 V1 chép: 'ĐÁP ÁN: A'
[bf16] B-train-00238-q08 gold=0  V0_goc=0 OK  V1_tung_buoc=0 OK  V2_bo_cot_dau=0 OK
    B-train-00740-q07 V1 chép: 'ĐÁP ÁN: A'
[bf16] B-train-00740-q07 gold=0  V0_goc=0 OK  V1_tung_buoc=0 OK  V2_bo_cot_dau=0 OK
    B-train-00850-q03 V1 chép: 'ĐÁ

In [13]:
# --- Cell 2d: SANITY 3 — model SO SÁNH được hay chỉ BÁM VỊ TRÍ? (chạy SAU 2c) ---
# Nhãn A/B nay VẼ LÊN ẢNH (khung + chữ cái ở cột trái), không còn chỉ nằm trong
# prompt. Đó là thứ bản chạy trước thiếu: prompt khẳng định "dải TRÊN là (A)"
# còn ảnh trống, nên 120/120 trả "A" không phân biệt được với "không so được nét".
#
# Hai phép đảo trên cùng ảnh, tách hai giả thuyết:
#   ĐẢO DẢI  (V3): đảo thứ tự, nhãn dính theo dải -> đáp án ĐÚNG không đổi.
#                  Chữ cái LẬT = model chọn theo VỊ TRÍ.
#   ĐỔI NHÃN (V4): giữ thứ tự, đổi chữ cái     -> đáp án đúng PHẢI đổi A->B.
#                  Chữ cái ĐỨNG = model không đọc nhãn trong ảnh.
#
# Chỉ V3 là không đủ: "LẬT" có thể do model nhìn thấy nhãn, mà cũng có thể do
# model đọc được độ đậm. V4 phân biệt hai cái đó.
import argparse, importlib
import probe_vlm_bold as P

P = importlib.reload(P)
P.OUT = P_OUT
P.cmd_sanity3(argparse.Namespace(n=10))


Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

[bf16] transformers 5.18.0 | dtype thật=torch.bfloat16
sanity3 [bf16]: 10 ảnh, đảo dải (nhãn dính theo dải) + đổi nhãn

[bf16] B-train-00780-q04 gold=1 gốc=0 đảo=0 đổi_nhãn=0
[bf16] B-train-00212-q01 gold=1 gốc=0 đảo=0 đổi_nhãn=0
[bf16] B-train-00594-q02 gold=1 gốc=0 đảo=0 đổi_nhãn=0
[bf16] B-train-00148-q04 gold=1 gốc=0 đảo=0 đổi_nhãn=0
[bf16] B-train-00576-q01 gold=1 gốc=0 đảo=0 đổi_nhãn=0
[bf16] B-train-00238-q08 gold=0 gốc=0 đảo=0 đổi_nhãn=0
[bf16] B-train-00740-q07 gold=0 gốc=0 đảo=0 đổi_nhãn=0
[bf16] B-train-00850-q03 gold=1 gốc=0 đảo=0 đổi_nhãn=0
[bf16] B-train-00466-q02 gold=0 gốc=0 đảo=0 đổi_nhãn=0
[bf16] B-train-01006-q09 gold=0 gốc=0 đảo=0 đổi_nhãn=0

[bf16] V0_goc       đúng 4/10
[bf16] V3_dao       đúng 4/10
[bf16] V4_doi_nhan  đúng 4/10

[bf16] ĐẢO DẢI   (nhãn dính theo dải) -> chữ cái lật: 0/10
[bf16] ĐỔI NHÃN  (giữ thứ tự)         -> chữ cái đứng: 10/10

Đọc (cả hai phép tính trên V0_goc, ảnh đúng nét chữ như nhau):
  ĐẢO DẢI lật  + ĐỔI NHÃN đứng -> model đọc nhãn trong

In [ ]:
# --- Cell 2e: SANITY 4 — nút thắt là PHÂN GIẢI hay ViT VỨT nét? (chạy SAU 2d) ---
# 2d kết luận âm: model không đọc nhãn trong ảnh, không so được nét. Hai nguyên
# nhân khác nhau, dẫn tới hai việc TRÁI NGƯỢC nhau:
#   (a) thiếu phân giải: dòng chữ 48 px chỉ trải 3,4 patch 14 px; chênh đậm-nhạt
#       1-2 px chìm trong một patch. -> sửa bằng BIỂU DIỄN ĐẦU VÀO (cắt sát dòng
#       rồi phóng to). QLoRA không cần.
#   (b) ViT vứt nét thật: QLoRA đóng băng ViT (mặc định) cũng vô ích, cả Cấu
#       hình D đổ. -> dừng, ghi kết luận âm.
#
# Phép thử: phóng canvas 4x, chạy LẠI đúng ba biến thể của 2d trên CÙNG 10 ảnh
# — VÀ chạy luôn 1x trong cùng lượt, cùng model, để phép so 1x/4x là so CẶP.
# (2d đã đo 1x rồi nhưng ở bf16; đem 4x-4bit so với 1x-bf16 là lệch hai biến.)
# ~4 phút.
#
# Chạy 4-bit chứ không bf16: 4x làm mỗi ảnh thành ~29k patch, activation ViT
# ~1,1 GB cộng trọng số 2 byte/param của 3B ~6 GB là tràn T4 14,56 GB — đúng cái
# OOM đã gặp. 4-bit hạ trọng số còn ~1,7 GB. Và 4-bit không đổi phép đo: 2b đã
# chạy 4-bit song song bf16, ra y hệt 10/10.
#
# Cell tự in lưới token 1x vs 4x và tự assert lưới phải nở — nếu `max_pixels`
# không ăn thì `smart_resize` thu ảnh về và probe đo nhầm thứ khác.
import argparse, importlib
import probe_vlm_bold as P

P = importlib.reload(P)
P.OUT = P_OUT
P.cmd_sanity4(argparse.Namespace(n=10))


In [14]:
# --- Cell 3: chạy Qwen2.5-VL-3B zero-shot ---
import argparse, importlib
import probe_vlm_bold as P

# Kernel giữ module cũ trong cache nếu Cell này đã chạy 1 lần -> nạp lại từ Drive.
P = importlib.reload(P)
P.OUT = P_OUT
P.cmd_run(argparse.Namespace(quant4=True))   # T4 16 GB: 4-bit cho chắc

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

[4-bit] transformers 5.18.0 | dtype thật=torch.bfloat16
[1/120] B-train-00780-q04 -> 'A' pick=0
[2/120] B-train-00212-q01 -> 'A' pick=0
[3/120] B-train-00594-q02 -> 'A' pick=0
[4/120] B-train-00148-q04 -> 'A' pick=0
[5/120] B-train-00576-q01 -> 'A' pick=0
[6/120] B-train-00238-q08 -> 'A' pick=0
[7/120] B-train-00740-q07 -> 'A' pick=0
[8/120] B-train-00850-q03 -> 'A' pick=0
[9/120] B-train-00466-q02 -> 'A' pick=0
[10/120] B-train-01006-q09 -> 'A' pick=0
[11/120] B-train-00026-q09 -> 'A' pick=0
[12/120] B-train-00162-q04 -> 'A' pick=0
[13/120] B-train-00864-q07 -> 'A' pick=0
[14/120] B-train-00350-q08 -> 'A' pick=0
[15/120] B-train-01008-q06 -> 'A' pick=0
[16/120] B-train-00022-q08 -> 'A' pick=0
[17/120] B-train-00202-q04 -> 'A' pick=0
[18/120] B-train-00268-q08 -> 'A' pick=0
[19/120] B-train-00872-q07 -> 'A' pick=0
[20/120] B-train-01018-q02 -> 'A' pick=0
[21/120] B-train-00714-q06 -> 'A' pick=0
[22/120] B-train-00666-q03 -> 'A' pick=0
[23/120] B-train-00790-q08 -> 'A' pick=0
[24/120] B

In [15]:
# --- Cell 4: kiểm tra kết quả đã về Drive ---
n = sum(1 for _ in open(P_OUT / "preds.jsonl", encoding="utf-8"))
print(f"{n} dự đoán -> {P_OUT/'preds.jsonl'}")
print("Về máy chạy:  python code/probe_vlm_bold.py score")

120 dự đoán -> /content/drive/MyDrive/Slot 1/Working Files/probe_d/preds.jsonl
Về máy chạy:  python code/probe_vlm_bold.py score
